# Rolling Model Evaluation Summary (Final Table)

## Objective
Build a final, structured summary table of the rolling training and testing evaluation results for all models.

This table consolidates the results of:

- Baseline (Naive model)
- XGBoost
- LightGBM

across all rolling iterations used in the backtesting strategy.

---

## Why This Step Matters
The rolling evaluation process generates multiple performance outputs across time.  
This notebook transforms those results into a **clean, standardized, and presentation-ready dataset**.

It enables:

- comparison of model performance across time
- identification of the best-performing model per iteration
- validation of model stability
- direct use in academic reporting and visualization

---

## Rolling Evaluation Framework

Each iteration follows:

- **Training window:** 90 days
- **Testing window:** 1 month ahead
- **Strategy:** rolling forward in time

For each iteration:

- model is trained on historical data
- model is evaluated on unseen future data

---

## Data Source

Evaluation results are loaded from: goldv2_netflow_direct_model_compare_weight10_v3_cyclic_stationtrend

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# ============================================================
# BUILD FINAL ROLLING TRAIN/TEST SUMMARY TABLE
# From already saved compare evaluation parquet
# ============================================================

EVAL_DIR_COMPARE = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/"
    "goldv2_netflow_direct_model_compare_weight10_v3_cyclic_stationtrend"
)

TRAIN_LOOKBACK_DAYS = 90

# -----------------------------
# 1) LOAD SAVED RESULTS
# -----------------------------
df_compare = spark.read.parquet(EVAL_DIR_COMPARE)

# Ensure order
df_compare = df_compare.orderBy("year", "month")

pdf = df_compare.toPandas().copy()

# -----------------------------
# 2) ADD ITERATION + DATE WINDOWS
# -----------------------------
pdf = pdf.sort_values(["year", "month"]).reset_index(drop=True)
pdf["iteration"] = range(1, len(pdf) + 1)

pdf["test_start"] = pd.to_datetime(
    pdf["year"].astype(str) + "-" + pdf["month"].astype(str).str.zfill(2) + "-01"
)

pdf["test_end"] = pdf["test_start"] + pd.offsets.MonthEnd(0)
pdf["train_end"] = pdf["test_start"] - pd.Timedelta(days=1)
pdf["train_start"] = pdf["train_end"] - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

# Format dates as strings
for c in ["train_start", "train_end", "test_start", "test_end"]:
    pdf[c] = pd.to_datetime(pdf[c]).dt.strftime("%Y-%m-%d")

# -----------------------------
# 3) REORDER COLUMNS
# -----------------------------
final_cols = [
    "iteration",
    "year",
    "month",
    "train_start",
    "train_end",
    "test_start",
    "test_end",
    "rows_test",
    "baseline_mae",
    "baseline_rmse",
    "baseline_r2",
    "xgb_mae",
    "xgb_rmse",
    "xgb_r2",
#    "xgb_best_n_estimators",
    "lgbm_mae",
    "lgbm_rmse",
    "lgbm_r2",
#    "lgbm_best_n_estimators",
    "winner_model"
#    "weight_event",
]

pdf_final = pdf[final_cols].copy()

# Optional rounding for presentation
metric_cols = [
    "baseline_mae", "baseline_rmse", "baseline_r2",
    "xgb_mae", "xgb_rmse", "xgb_r2",
    "lgbm_mae", "lgbm_rmse", "lgbm_r2"
]

for c in metric_cols:
    pdf_final[c] = pdf_final[c].astype(float).round(4)

# -----------------------------
# 4) DISPLAY
# -----------------------------
display(pdf_final)

# -----------------------------
# 5) SAVE FINAL TABLE
# -----------------------------
FINAL_SUMMARY_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/"
    "goldv2_netflow_direct_rolling_summary_for_tutor"
)

spark.createDataFrame(pdf_final).write.mode("overwrite").parquet(FINAL_SUMMARY_DIR)

print("Saved final tutor summary to:", FINAL_SUMMARY_DIR)

# Rolling Model Evaluation Summary Completed

## Summary of Results

- Total iterations evaluated: **23**
- Time coverage: **Oct 2022 – Sep 2024**
- Evaluation strategy: **Rolling backtesting (90-day training + 1-month testing)**

---

## Model Performance Overview

Across the rolling evaluation:

- Machine learning models (XGBoost and LightGBM) consistently outperform the baseline
- Performance is stable across time with expected seasonal variation
- Both models show strong predictive capability for station imbalance

---

## Best Model Selection

The best model per iteration is determined based on predictive performance (primarily MAE).

Observations:

- **XGBoost** is frequently the top-performing model
- **LightGBM** also performs competitively in several periods
- Model dominance varies slightly depending on temporal patterns

---

## Baseline vs ML Models

- Baseline model shows higher error and lower explanatory power
- ML models significantly reduce MAE and RMSE
- R² improves from negative/low values (baseline) to positive values (ML models)

This provides strong empirical support for the hypothesis:

> External features and advanced models improve prediction of station imbalance.

---

## Temporal Consistency

- Model performance remains consistent across multiple months
- Some variability exists due to:
  - seasonality
  - demand fluctuations
  - event-driven spikes

However, ML models remain robust across these conditions.

---

## Output Dataset

Final summary stored at: goldv2_netflow_direct_rolling_summary_for_tutor

# Rolling Model Performance (MAE)

## Objective
Compare predictive performance across 23 rolling backtesting iterations using **Mean Absolute Error (MAE)**.

Models evaluated:
- Naive Baseline (persistence model)
- XGBoost
- LightGBM

---

## Metric: MAE

MAE measures the average absolute difference between predicted and actual net flow:

- Lower MAE → better predictive accuracy  
- Directly interpretable in operational terms (bike imbalance units)

---

## Rolling Backtesting Design

Each iteration represents:

- Training window: last 90 days  
- Testing window: next month  

This simulates real-world forecasting conditions and ensures no data leakage.

---

## X-Axis Interpretation

- Each point corresponds to a **monthly test period (YYYY-MM)**  
- Total: 23 sequential iterations  

---

## Expected Behavior

If external features are useful:

- ML models should consistently outperform the baseline  
- Error patterns may follow seasonal demand trends  

In [0]:
import matplotlib.pyplot as plt

# -----------------------------
# PREPARAR LABELS
# -----------------------------
plot_df = pdf_final.copy()
plot_df["label"] = plot_df["year"].astype(str) + "-" + plot_df["month"].astype(str).str.zfill(2)

# -----------------------------
# GRÁFICO MAE
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    plot_df["iteration"],
    plot_df["baseline_mae"],
    marker="o",
    linewidth=2,
    label="Naive Baseline (MAE)"
)

ax.plot(
    plot_df["iteration"],
    plot_df["xgb_mae"],
    marker="o",
    linewidth=2,
    label="XGBoost (MAE)"
)

ax.plot(
    plot_df["iteration"],
    plot_df["lgbm_mae"],
    marker="o",
    linewidth=2,
    label="LightGBM (MAE)"
)

# Eje X
ax.set_xticks(plot_df["iteration"])
ax.set_xticklabels(plot_df["label"], rotation=45)

# Títulos
ax.set_title("Rolling Backtesting Performance Across 23 Iterations (MAE)", fontsize=14)
ax.set_xlabel("Test Month / Iteration")
ax.set_ylabel("MAE")

# Grid
ax.grid(True, linestyle="--", alpha=0.5)

# Leyenda
ax.legend(frameon=True)

plt.tight_layout()
plt.show()

# Results: Rolling MAE Analysis

## Key Observations

- Both **XGBoost** and **LightGBM** consistently achieve lower MAE than the Naive baseline  
- The performance gap is **large and stable across all iterations**  
- XGBoost and LightGBM show **very similar behavior over time**

---

## Baseline vs ML Models

- The Naive model exhibits significantly higher MAE, especially during:
  - mid-year (summer demand peaks)
  - high-variability periods  

- ML models reduce error substantially, indicating:
  - better adaptation to demand fluctuations  
  - improved generalization  

---

## Temporal Patterns

The MAE curves reveal strong **seasonality effects**:

- Higher errors in:
  - summer months (higher demand variability)  
- Lower errors in:
  - winter months (more stable demand patterns)  

Despite this, ML models remain consistently more accurate.

---

## XGBoost vs LightGBM

- Both models perform nearly identically  
- XGBoost shows slightly better performance in some periods  
- LightGBM remains highly competitive and stable  

---

## Hypothesis Interpretation

The consistent improvement over the baseline supports:

> **H1: Features have a statistically significant effect on predicting station imbalance risk**

Because:

- MAE is lower in **all iterations**
- Improvements are consistent across time
- Models capture temporal + external patterns beyond persistence

---

## Key Insight

This visualization demonstrates that:

- External features (weather, events, temporal signals) significantly enhance predictive performance  
- Machine learning models provide **robust and reliable forecasts**  
- The system is suitable for operational decision-making in bike-share systems  

---

# Model Improvement vs Naive Baseline

## Objective
Quantify how much machine learning models improve prediction accuracy over a Naive baseline.

---

## Metric: Improvement (%)

Improvement is computed as:

\[
Improvement = \frac{MAE_{baseline} - MAE_{model}}{MAE_{baseline}} \times 100
\]

- Positive values → model outperforms baseline  
- Negative values → model underperforms baseline  

---

## Evaluation Setup

- Rolling backtesting (23 iterations)
- Training window: 90 days  
- Testing window: next month  

Each point represents one out-of-sample evaluation.

---

## Interpretation Guide

- Values above 0% indicate performance gain  
- Stability across iterations indicates robustness  
- Consistent gains suggest meaningful feature contribution  

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# PREPARAR DATA
# -----------------------------
plot_df = pdf_final.copy()

# Improvement % vs baseline usando MAE
plot_df["xgb_improve_pct"] = (
    (plot_df["baseline_mae"] - plot_df["xgb_mae"]) / plot_df["baseline_mae"]
) * 100

plot_df["lgbm_improve_pct"] = (
    (plot_df["baseline_mae"] - plot_df["lgbm_mae"]) / plot_df["baseline_mae"]
) * 100

# -----------------------------
# BONUS: MÉTRICAS RESUMEN
# -----------------------------
xgb_avg_improve = plot_df["xgb_improve_pct"].mean()
lgbm_avg_improve = plot_df["lgbm_improve_pct"].mean()

xgb_win_pct = (plot_df["xgb_improve_pct"] > 0).mean() * 100
lgbm_win_pct = (plot_df["lgbm_improve_pct"] > 0).mean() * 100

print(f"XGBoost average improvement vs baseline: {xgb_avg_improve:.2f}%")
print(f"LightGBM average improvement vs baseline: {lgbm_avg_improve:.2f}%")
print(f"XGBoost improves over baseline in: {xgb_win_pct:.1f}% of iterations")
print(f"LightGBM improves over baseline in: {lgbm_win_pct:.1f}% of iterations")

# -----------------------------
# DEFINIR RANGO Y MÁS COMPACTO
# -----------------------------
all_vals = np.concatenate([
    plot_df["xgb_improve_pct"].values,
    plot_df["lgbm_improve_pct"].values
])

y_min = np.floor(all_vals.min()) - 1
y_max = np.ceil(all_vals.max()) + 1

# Si todos son positivos, igual deja una línea 0 visible cerca del borde inferior
if y_min > 0:
    y_min = -1

# -----------------------------
# PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 3.3))

ax.plot(
    plot_df["iteration"],
    plot_df["xgb_improve_pct"],
    marker="o",
    linewidth=2,
    markersize=5,
    label="XGBoost Improvement (%)"
)

ax.plot(
    plot_df["iteration"],
    plot_df["lgbm_improve_pct"],
    marker="o",
    linewidth=2,
    markersize=5,
    label="LightGBM Improvement (%)"
)

# Línea base
ax.axhline(0, linestyle="--", linewidth=1, alpha=0.8)

# -----------------------------
# FORMATO
# -----------------------------
ax.set_title("MAE Improvement vs Naive Baseline (23 Rolling Iterations)", fontsize=12)
ax.set_xlabel("Iteration")
ax.set_ylabel("Improvement (%)")

ax.set_xticks(plot_df["iteration"])

# Compactar visualmente el eje Y
ax.set_ylim(y_min, y_max)

ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(frameon=True, loc="best")

plt.tight_layout()
plt.show()

# Results: Model Improvement Analysis

## Key Findings

- **XGBoost average improvement:** ~30.6%  
- **LightGBM average improvement:** ~30.4%  
- Both models outperform the baseline in **100% of iterations**

---

## Performance Behavior

- Improvement remains consistently between **~28% and ~33%**
- No negative values observed → models always outperform baseline  
- Very low variability → highly stable performance  

---

## Model Comparison

- XGBoost and LightGBM show **almost identical performance**
- XGBoost slightly leads in some iterations  
- Both models are robust and reliable  

---

## Hypothesis Testing

These results provide strong evidence to:

> **Reject H₀ and support H₁**

Because:

- Improvement is **consistently positive across all iterations**
- Gains are **substantial (~30%)**
- Performance is **stable over time**

---

## Key Insight

External features (temporal, weather, events):

- Significantly improve prediction accuracy  
- Add consistent value beyond persistence models  
- Enable more reliable forecasting for operational decision-making  

---

## Takeaway

Machine learning models reduce prediction error by ~30%,  
demonstrating the **critical importance of enriched feature engineering**.